# 🎵 MichSet Audio Extraction for Google Colab

**Complete MichSetHowusu Audio Processing Pipeline**

This notebook extracts audio files from MichSet Arrow datasets and prepares them for TTS/ASR training.

## 🎯 What this notebook does:
- ✅ Installs FFmpeg and TorchCodec with proper compatibility
- ✅ Extracts audio from Twi, Akuapem, and Fante datasets
- ✅ Creates TTS-ready metadata files
- ✅ Optimized for Google Colab environment
- ✅ Handles all common error cases gracefully

## 📋 Prerequisites:
1. Google Drive mounted with MichSet datasets
2. Datasets located at: `/content/drive/My Drive/voice-lab-data/raw/michsethowusu/`
3. Expected datasets: `twi_multispeaker_audio_transcribed`, `akuapem_multispeaker_audio_transcribed`, `fante_multispeaker_audio_transcribed`

---

## 🔧 Step 1: Environment Setup & Google Drive Mount

In [ ]:
# Mount Google Drive
from google.colab import drive
import os
from pathlib import Path

print("🔗 Mounting Google Drive...")
drive.mount('/content/drive')

# Setup paths
BASE_DIR = Path('/content/drive/My Drive/voice-lab-data')
MICHSET_DIR = BASE_DIR / 'raw' / 'michsethowusu'
OUTPUT_DIR = BASE_DIR / 'processed' / 'michset_audio_extracted'

print(f"📁 Base directory: {BASE_DIR}")
print(f"📁 MichSet directory: {MICHSET_DIR}")
print(f"📁 Output directory: {OUTPUT_DIR}")

# Verify MichSet datasets exist
datasets = ['twi_multispeaker_audio_transcribed', 'akuapem_multispeaker_audio_transcribed', 'fante_multispeaker_audio_transcribed']
missing_datasets = []

for dataset in datasets:
    dataset_path = MICHSET_DIR / dataset
    if dataset_path.exists():
        print(f"✅ Found: {dataset}")
    else:
        print(f"❌ Missing: {dataset}")
        missing_datasets.append(dataset)

if missing_datasets:
    print(f"\n⚠️ Warning: {len(missing_datasets)} datasets not found")
    print("💡 Make sure MichSet datasets are uploaded to Google Drive at the expected location")
else:
    print("\n🎉 All MichSet datasets found! Ready to proceed.")

## 🔧 Step 2: FFmpeg Installation & Verification

In [ ]:
import subprocess
import sys

def install_ffmpeg():
    """Install FFmpeg via apt package manager."""
    print("🔧 Installing FFmpeg...")
    
    # Update package list
    print("📦 Updating package list...")
    result = subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
    
    # Install FFmpeg
    print("📦 Installing FFmpeg...")
    result = subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], capture_output=True)
    
    if result.returncode == 0:
        print("✅ FFmpeg installed successfully")
        return True
    else:
        print(f"❌ FFmpeg installation failed: {result.stderr.decode()[:200]}")
        return False

def verify_ffmpeg():
    """Verify FFmpeg installation and get version info."""
    try:
        result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
        if result.returncode == 0:
            version_line = result.stdout.split('\n')[0]
            print(f"✅ FFmpeg verified: {version_line}")
            return True
        else:
            print("❌ FFmpeg verification failed")
            return False
    except FileNotFoundError:
        print("❌ FFmpeg not found in PATH")
        return False

# Install and verify FFmpeg
print("🚀 FFMPEG SETUP")
print("=" * 40)

ffmpeg_ok = install_ffmpeg()
if ffmpeg_ok:
    ffmpeg_verified = verify_ffmpeg()
    if ffmpeg_verified:
        print("\n🎉 FFmpeg is ready for TorchCodec integration!")
    else:
        print("\n⚠️ FFmpeg installed but verification failed")
else:
    print("\n❌ FFmpeg setup failed - cannot proceed with audio extraction")

## 🔧 Step 3: TorchCodec Installation with Multiple Fallbacks

In [ ]:
def install_torchcodec_comprehensive():
    """Install TorchCodec with comprehensive fallback strategies."""
    
    print("🚀 TORCHCODEC INSTALLATION")
    print("=" * 40)
    
    # Check if already installed
    try:
        import torchcodec
        print(f"✅ TorchCodec already available: {torchcodec.__version__}")
        return True
    except ImportError:
        pass
    
    # Method 1: Direct pip install
    print("📦 Method 1: Standard pip installation...")
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            "torchcodec", "--quiet", "--no-cache-dir"
        ])
        
        import torchcodec
        print(f"✅ Method 1 success: TorchCodec {torchcodec.__version__}")
        return True
        
    except Exception as e:
        print(f"⚠️ Method 1 failed: {str(e)[:100]}")
    
    # Method 2: Install with specific PyTorch version
    print("📦 Method 2: PyTorch-compatible installation...")
    try:
        # First ensure PyTorch is up to date
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            "torch>=2.0.0", "torchaudio>=2.0.0", "--quiet"
        ])
        
        # Then install torchcodec
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            "torchcodec", "--quiet", "--no-cache-dir", "--force-reinstall"
        ])
        
        import torchcodec
        print(f"✅ Method 2 success: TorchCodec {torchcodec.__version__}")
        return True
        
    except Exception as e:
        print(f"⚠️ Method 2 failed: {str(e)[:100]}")
    
    # Method 3: Try development version
    print("📦 Method 3: Development version installation...")
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            "--pre", "torchcodec", "--quiet", "--no-cache-dir"
        ])
        
        import torchcodec
        print(f"✅ Method 3 success: TorchCodec {torchcodec.__version__}")
        return True
        
    except Exception as e:
        print(f"⚠️ Method 3 failed: {str(e)[:100]}")
    
    print("❌ All TorchCodec installation methods failed")
    return False

def test_torchcodec_functionality():
    """Test if TorchCodec can work with FFmpeg."""
    print("\n🧪 Testing TorchCodec functionality...")
    
    try:
        import torchcodec
        from torchcodec.decoders import VideoDecoder
        
        print("✅ TorchCodec imports successful")
        print(f"📋 TorchCodec version: {torchcodec.__version__}")
        
        # Try to create a decoder instance (this tests FFmpeg integration)
        # We can't test with actual video without a file, but import success is good
        print("✅ TorchCodec FFmpeg integration appears functional")
        return True
        
    except RuntimeError as e:
        error_msg = str(e)
        if "Could not load libtorchcodec" in error_msg:
            print("❌ TorchCodec cannot load FFmpeg libraries")
            print("💡 This usually means FFmpeg libraries are incompatible")
            print("💡 Try: Runtime → Restart Runtime, then re-run all cells")
        else:
            print(f"⚠️ TorchCodec runtime error: {error_msg[:200]}")
        return False
        
    except Exception as e:
        print(f"❌ TorchCodec test failed: {str(e)[:200]}")
        return False

# Run TorchCodec installation
torchcodec_installed = install_torchcodec_comprehensive()

if torchcodec_installed:
    torchcodec_functional = test_torchcodec_functionality()
    
    if torchcodec_functional:
        print("\n🎉 TORCHCODEC READY FOR AUDIO EXTRACTION!")
    else:
        print("\n⚠️ TorchCodec installed but may have FFmpeg compatibility issues")
        print("💡 Recommendation: Restart runtime and try again")
else:
    print("\n❌ TorchCodec installation failed - audio extraction will not work")

## 📦 Step 4: Install Additional Audio Processing Dependencies

In [ ]:
# Install required audio processing libraries
print("📦 Installing audio processing libraries...")

required_packages = [
    "librosa",
    "soundfile", 
    "datasets",
    "pandas",
    "numpy",
    "tqdm",
    "pydub"
]

for package in required_packages:
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            package, "--quiet", "--upgrade"
        ])
        print(f"✅ {package}")
    except Exception as e:
        print(f"⚠️ {package}: {str(e)[:50]}")

# Import all required libraries
print("\n📚 Importing libraries...")
try:
    import librosa
    import soundfile as sf
    import pandas as pd
    import numpy as np
    from datasets import load_dataset, Audio
    from tqdm.auto import tqdm
    import json
    import warnings
    warnings.filterwarnings('ignore')
    
    print("✅ All libraries imported successfully")
    print(f"📋 Librosa version: {librosa.__version__}")
    print(f"📋 Pandas version: {pd.__version__}")
    print(f"📋 NumPy version: {np.__version__}")
    
except Exception as e:
    print(f"❌ Library import failed: {e}")
    print("💡 Try restarting runtime and re-running cells")

## 🎯 Step 5: MichSet Audio Extractor Class Definition

In [ ]:
class ColabMichSetExtractor:
    """Google Colab optimized MichSet audio extractor."""
    
    def __init__(self, output_dir: Path, target_sr: int = 16000):
        self.output_dir = Path(output_dir)
        self.target_sr = target_sr
        self.stats = {
            'total_processed': 0,
            'total_extracted': 0,
            'total_failed': 0,
            'total_duration': 0.0
        }
        
        # Create output directory
        self.output_dir.mkdir(parents=True, exist_ok=True)
    
    def extract_single_dataset(self, dataset_path: Path, dataset_name: str, max_samples: int = 1000):
        """Extract audio from a single MichSet dataset."""
        
        print(f"\n🎵 PROCESSING: {dataset_name}")
        print("=" * 50)
        
        if not dataset_path.exists():
            print(f"❌ Dataset not found: {dataset_path}")
            return {'status': 'missing', 'extracted': 0}
        
        try:
            # Load dataset
            print(f"📂 Loading dataset from {dataset_path}...")
            dataset = load_dataset(str(dataset_path), split="train")
            
            total_samples = min(len(dataset), max_samples)
            print(f"📊 Dataset size: {len(dataset)} samples")
            print(f"📊 Processing: {total_samples} samples (max: {max_samples})")
            print(f"📋 Features: {list(dataset.features.keys())}")
            
            # Setup audio decoding
            if 'audio_path' in dataset.features:
                print(f"🔄 Configuring audio decoding for 'audio_path' field...")
                dataset = dataset.cast_column('audio_path', Audio(sampling_rate=self.target_sr))
            elif 'audio' in dataset.features:
                print(f"🔄 Configuring audio decoding for 'audio' field...")
                dataset = dataset.cast_column('audio', Audio(sampling_rate=self.target_sr))
            else:
                print(f"⚠️ No recognized audio field found in dataset features")
                return {'status': 'no_audio_field', 'extracted': 0}
            
            # Create dataset output directory
            dataset_output_dir = self.output_dir / dataset_name
            dataset_output_dir.mkdir(parents=True, exist_ok=True)
            
            extracted_files = []
            failed_count = 0
            
            # Process samples with progress bar
            for i in tqdm(range(total_samples), desc=f"Extracting {dataset_name}", unit="samples"):
                try:
                    sample = dataset[i]
                    self.stats['total_processed'] += 1
                    
                    # Get transcription
                    transcription = sample.get('transcription', '').strip()
                    if not transcription or len(transcription) < 2:
                        continue
                    
                    # Get audio data
                    audio_data = sample.get('audio_path') or sample.get('audio')
                    if not audio_data:
                        continue
                    
                    # Extract audio array and sample rate
                    if isinstance(audio_data, dict) and 'array' in audio_data:
                        audio_array = np.array(audio_data['array'], dtype=np.float32)
                        sample_rate = audio_data.get('sampling_rate', self.target_sr)
                    else:
                        continue
                    
                    # Resample if necessary
                    if sample_rate != self.target_sr:
                        audio_array = librosa.resample(
                            audio_array, 
                            orig_sr=sample_rate, 
                            target_sr=self.target_sr
                        )
                    
                    # Quality checks
                    duration = len(audio_array) / self.target_sr
                    max_amplitude = np.max(np.abs(audio_array))
                    
                    # Skip poor quality audio
                    if duration < 0.3 or duration > 30.0 or max_amplitude < 0.001:
                        continue
                    
                    # Normalize audio
                    if max_amplitude > 0:
                        audio_array = audio_array / max_amplitude * 0.95
                    
                    # Save audio file
                    output_filename = f"{dataset_name}_{i:05d}.wav"
                    output_path = dataset_output_dir / output_filename
                    
                    sf.write(str(output_path), audio_array, self.target_sr)
                    
                    # Record extraction info
                    extracted_files.append({
                        'audio_file': str(output_path),
                        'transcription': transcription,
                        'duration': duration,
                        'dataset': dataset_name,
                        'sample_index': i,
                        'max_amplitude': max_amplitude
                    })
                    
                    self.stats['total_extracted'] += 1
                    self.stats['total_duration'] += duration
                    
                except Exception as e:
                    failed_count += 1
                    self.stats['total_failed'] += 1
                    
                    # Show first few errors for debugging
                    if failed_count <= 3:
                        print(f"    ⚠️ Sample {i} failed: {str(e)[:80]}")
                    
                    continue
            
            # Save results if any files were extracted
            if extracted_files:
                return self._save_dataset_results(dataset_name, dataset_output_dir, extracted_files, failed_count)
            else:
                print(f"❌ No audio files extracted (failed: {failed_count})")
                return {'status': 'no_files_extracted', 'extracted': 0, 'failed': failed_count}
                
        except Exception as e:
            error_msg = str(e)
            print(f"❌ Dataset processing error: {error_msg[:300]}")
            
            # Provide specific troubleshooting guidance
            if 'torchcodec' in error_msg.lower():
                print("💡 TorchCodec Error - Try:")
                print("   1. Runtime → Restart Runtime")
                print("   2. Re-run FFmpeg and TorchCodec setup cells")
                print("   3. Check that FFmpeg installation completed successfully")
            elif 'memory' in error_msg.lower():
                print("💡 Memory Error - Try:")
                print("   1. Reduce max_samples parameter")
                print("   2. Runtime → Factory Reset Runtime")
            elif 'drive' in error_msg.lower():
                print("💡 Drive Error - Try:")
                print("   1. Re-mount Google Drive")
                print("   2. Check dataset file permissions")
            
            return {'status': 'error', 'error': error_msg, 'extracted': 0}
    
    def _save_dataset_results(self, dataset_name: str, output_dir: Path, extracted_files: list, failed_count: int):
        """Save extraction results and create metadata files."""
        
        # Create DataFrame
        df = pd.DataFrame(extracted_files)
        
        # Save detailed metadata
        metadata_path = output_dir / 'extracted_metadata.csv'
        df.to_csv(metadata_path, index=False, encoding='utf-8')
        
        # Create TTS metadata.txt format
        tts_lines = []
        for _, row in df.iterrows():
            text = str(row['transcription']).replace('|', '-').replace('\n', ' ').strip()
            speaker_id = f"{dataset_name}_speaker"
            tts_lines.append(f"{row['audio_file']}|{text}|{speaker_id}")
        
        tts_metadata_path = output_dir / 'metadata.txt'
        with open(tts_metadata_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(tts_lines))
        
        # Calculate statistics
        avg_duration = df['duration'].mean()
        total_duration = df['duration'].sum()
        
        print(f"✅ Successfully extracted {len(extracted_files)} audio files")
        print(f"📊 Average duration: {avg_duration:.2f}s")
        print(f"🕐 Total duration: {total_duration/60:.1f} minutes")
        print(f"💾 Metadata: {metadata_path}")
        print(f"📄 TTS format: {tts_metadata_path}")
        
        if failed_count > 0:
            success_rate = (len(extracted_files) / (len(extracted_files) + failed_count)) * 100
            print(f"⚠️ Failed extractions: {failed_count} (success rate: {success_rate:.1f}%)")
        
        return {
            'status': 'success',
            'extracted': len(extracted_files),
            'failed': failed_count,
            'avg_duration': avg_duration,
            'total_duration': total_duration,
            'metadata_file': str(metadata_path),
            'tts_metadata_file': str(tts_metadata_path),
            'output_dir': str(output_dir)
        }
    
    def process_all_datasets(self, base_dir: Path, max_samples_per_dataset: int = 1000):
        """Process all MichSet datasets."""
        
        print("🚀 MICHSET AUDIO EXTRACTION FOR GOOGLE COLAB")
        print("=" * 60)
        
        datasets = [
            ('twi_multispeaker_audio_transcribed', 'twi'),
            ('akuapem_multispeaker_audio_transcribed', 'akuapem_twi'),
            ('fante_multispeaker_audio_transcribed', 'fante')
        ]
        
        results = {}
        
        for dataset_name, dialect in datasets:
            dataset_path = base_dir / dataset_name
            result = self.extract_single_dataset(dataset_path, dataset_name, max_samples_per_dataset)
            results[dataset_name] = result
        
        # Create comprehensive summary
        summary = self._create_extraction_summary(results)
        
        # Save summary
        summary_file = self.output_dir / 'michset_extraction_summary.json'
        with open(summary_file, 'w', encoding='utf-8') as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)
        
        return results, summary
    
    def _create_extraction_summary(self, results: dict):
        """Create comprehensive extraction summary."""
        
        successful_datasets = [name for name, result in results.items() 
                              if result.get('extracted', 0) > 0]
        
        summary = {
            'extraction_timestamp': pd.Timestamp.now().isoformat(),
            'total_datasets_processed': len(results),
            'successful_datasets': len(successful_datasets),
            'dataset_results': results,
            'overall_stats': self.stats.copy(),
            'colab_optimized': True,
            'ready_for_tts_training': len(successful_datasets) > 0
        }
        
        return summary

print("🎯 ColabMichSetExtractor class defined!")
print("📋 Ready to process MichSet datasets with full error handling and progress tracking")

## 🎵 Step 6: Run MichSet Audio Extraction

In [ ]:
# Initialize extractor
extractor = ColabMichSetExtractor(
    output_dir=OUTPUT_DIR,
    target_sr=16000  # 16kHz is standard for speech
)

# Run extraction with Colab-optimized settings
print(f"🎯 Starting extraction...")
print(f"📁 Input: {MICHSET_DIR}")
print(f"📁 Output: {OUTPUT_DIR}")
print(f"⚙️ Target sample rate: 16000 Hz")
print(f"⚙️ Max samples per dataset: 750 (Colab optimized)")

# Process all datasets
results, summary = extractor.process_all_datasets(
    base_dir=MICHSET_DIR,
    max_samples_per_dataset=750  # Colab-friendly limit
)

# Display final results
print("\n" + "=" * 60)
print("🎉 EXTRACTION COMPLETE!")
print("=" * 60)

print(f"📊 OVERALL STATISTICS:")
print(f"  Total samples processed: {summary['overall_stats']['total_processed']}")
print(f"  Total audio files extracted: {summary['overall_stats']['total_extracted']}")
print(f"  Total failed extractions: {summary['overall_stats']['total_failed']}")
print(f"  Total audio duration: {summary['overall_stats']['total_duration']/3600:.2f} hours")

print(f"\n📋 DATASET RESULTS:")
for dataset_name, result in results.items():
    status = result.get('status', 'unknown')
    extracted = result.get('extracted', 0)
    
    if status == 'success':
        duration_min = result.get('total_duration', 0) / 60
        print(f"  ✅ {dataset_name}: {extracted} files ({duration_min:.1f} min)")
    elif status == 'missing':
        print(f"  ❌ {dataset_name}: Dataset not found")
    elif status == 'error':
        print(f"  ❌ {dataset_name}: Error - {result.get('error', 'unknown')[:50]}")
    else:
        print(f"  ⚠️ {dataset_name}: {status}")

if summary['overall_stats']['total_extracted'] > 0:
    print(f"\n🎯 SUCCESS SUMMARY:")
    print(f"  ✅ {summary['successful_datasets']}/{summary['total_datasets_processed']} datasets processed successfully")
    print(f"  📁 Audio files saved to: {OUTPUT_DIR}")
    print(f"  📄 Each dataset has TTS-ready metadata.txt files")
    print(f"  📊 Detailed CSV metadata for analysis")
    print(f"  📋 Summary saved to: {OUTPUT_DIR}/michset_extraction_summary.json")
    
    print(f"\n🚀 NEXT STEPS:")
    print(f"  1. Use the extracted audio files in TTS training notebooks")
    print(f"  2. Each dataset folder contains:")
    print(f"     - WAV audio files")
    print(f"     - metadata.txt (TTS format: audio_path|text|speaker)")
    print(f"     - extracted_metadata.csv (detailed analysis data)")
    print(f"  3. Files are ready for frameworks like Coqui TTS, ESPnet, etc.")
else:
    print(f"\n❌ NO FILES EXTRACTED")
    print(f"💡 Troubleshooting steps:")
    print(f"  1. Check that MichSet datasets exist in Google Drive")
    print(f"  2. Restart runtime and re-run all setup cells")
    print(f"  3. Verify FFmpeg and TorchCodec installation success messages")
    print(f"  4. Check error messages above for specific issues")

## ✅ Step 7: Verify Extraction Results

In [ ]:
# Verify the extracted files
print("🔍 VERIFYING EXTRACTION RESULTS")
print("=" * 40)

datasets = ['twi_multispeaker_audio_transcribed', 'akuapem_multispeaker_audio_transcribed', 'fante_multispeaker_audio_transcribed']

total_files = 0
total_metadata_files = 0

for dataset_name in datasets:
    dataset_dir = OUTPUT_DIR / dataset_name
    
    if dataset_dir.exists():
        # Count audio files
        wav_files = list(dataset_dir.glob('*.wav'))
        metadata_txt = dataset_dir / 'metadata.txt'
        metadata_csv = dataset_dir / 'extracted_metadata.csv'
        
        print(f"\n📁 {dataset_name}:")
        print(f"  🎵 Audio files: {len(wav_files)}")
        print(f"  📄 TTS metadata: {'✅' if metadata_txt.exists() else '❌'}")
        print(f"  📊 CSV metadata: {'✅' if metadata_csv.exists() else '❌'}")
        
        if len(wav_files) > 0:
            # Check a sample file
            sample_file = wav_files[0]
            try:
                audio_data, sr = sf.read(str(sample_file))
                duration = len(audio_data) / sr
                print(f"  🔍 Sample check: {sample_file.name} ({duration:.2f}s, {sr}Hz)")
            except Exception as e:
                print(f"  ⚠️ Sample check failed: {str(e)[:50]}")
        
        total_files += len(wav_files)
        if metadata_txt.exists():
            total_metadata_files += 1
    else:
        print(f"\n❌ {dataset_name}: No output directory found")

print(f"\n📊 VERIFICATION SUMMARY:")
print(f"  Total audio files: {total_files}")
print(f"  Datasets with metadata: {total_metadata_files}/3")

if total_files > 0:
    print(f"\n✅ Extraction verification successful!")
    print(f"🎯 Files are ready for TTS/ASR training")
else:
    print(f"\n❌ No files found - extraction may have failed")
    print(f"💡 Check the extraction step output for error details")

## 📊 Step 8: Sample Data Analysis (Optional)

In [ ]:
# Analyze extracted samples
print("📊 SAMPLE DATA ANALYSIS")
print("=" * 40)

for dataset_name in ['twi_multispeaker_audio_transcribed', 'akuapem_multispeaker_audio_transcribed', 'fante_multispeaker_audio_transcribed']:
    metadata_file = OUTPUT_DIR / dataset_name / 'extracted_metadata.csv'
    
    if metadata_file.exists():
        try:
            df = pd.read_csv(metadata_file)
            
            print(f"\n📋 {dataset_name.upper()}:")
            print(f"  Samples: {len(df)}")
            print(f"  Avg duration: {df['duration'].mean():.2f}s")
            print(f"  Duration range: {df['duration'].min():.2f}s - {df['duration'].max():.2f}s")
            print(f"  Total duration: {df['duration'].sum()/60:.1f} minutes")
            
            # Show sample transcriptions
            print(f"  Sample texts:")
            for i, text in enumerate(df['transcription'].head(3)):
                print(f"    {i+1}. {text[:60]}{'...' if len(text) > 60 else ''}")
                
        except Exception as e:
            print(f"\n⚠️ {dataset_name}: Analysis failed - {str(e)[:50]}")
    else:
        print(f"\n❌ {dataset_name}: No metadata file found")

# Check TTS metadata format
print(f"\n📄 TTS METADATA FORMAT SAMPLE:")
for dataset_name in ['twi_multispeaker_audio_transcribed']:
    tts_file = OUTPUT_DIR / dataset_name / 'metadata.txt'
    if tts_file.exists():
        with open(tts_file, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            print(f"  📋 {dataset_name} (showing first 3 lines):")
            for i, line in enumerate(lines[:3]):
                print(f"    {i+1}. {line.strip()}")
            break

## 🎉 Completion Summary

### 🎯 What This Notebook Accomplished:

1. **✅ Environment Setup**: Mounted Google Drive and verified dataset locations
2. **✅ FFmpeg Installation**: Installed and verified FFmpeg for audio processing
3. **✅ TorchCodec Setup**: Installed TorchCodec with multiple fallback methods
4. **✅ Audio Extraction**: Extracted audio from MichSet Arrow datasets
5. **✅ Metadata Generation**: Created TTS-compatible metadata files
6. **✅ Quality Verification**: Verified extraction results and file integrity

### 📁 Output Structure:
```
/content/drive/My Drive/voice-lab-data/processed/michset_audio_extracted/
├── twi_multispeaker_audio_transcribed/
│   ├── *.wav (audio files)
│   ├── metadata.txt (TTS format)
│   └── extracted_metadata.csv (detailed data)
├── akuapem_multispeaker_audio_transcribed/
│   └── (same structure)
├── fante_multispeaker_audio_transcribed/
│   └── (same structure)
└── michset_extraction_summary.json
```

### 🚀 Next Steps:
1. **Use for TTS Training**: The extracted files are ready for Coqui TTS, ESPnet, or other frameworks
2. **ASR Training**: Audio files can also be used for automatic speech recognition
3. **Data Analysis**: Use the CSV metadata files for linguistic and acoustic analysis

### 💡 Troubleshooting:
- **If extraction failed**: Restart runtime and re-run all cells
- **If files are missing**: Check Google Drive permissions and dataset locations
- **If audio quality issues**: Adjust the quality thresholds in the extractor class

---
**🎵 MichSet Audio Extraction Complete!**